In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import switch_cwd_to_root

switch_cwd_to_root()

import nichepca as npc
import scanpy as sc
from jenkspy import JenksNaturalBreaks

In [ ]:
path = "data/xenium/processed/05.1-kidney_tcr_nichepca.h5ad"
adata = sc.read_h5ad(path)
adata

## Density-based approach

In [ ]:
# calc t cell density per cell neighborhood
npc.gc.construct_multi_sample_graph(
    adata, sample_key="sample", radius=50, remove_self_loops=False
)

In [ ]:
# perform graph aggregation
import numpy as np
import pandas as pd

obs_key = "cell_type_l1"
backend = "sparse"
aggr = "mean"

df = pd.get_dummies(adata.obs[obs_key], dtype=np.int8)
X = df.values
var = pd.DataFrame(index=df.columns)

ad_tmp = sc.AnnData(
    X=X,
    obs=adata.obs,
    var=var,
    uns=adata.uns,
)
npc.ne.aggregate(ad_tmp, backend=backend, aggr=aggr, obsm_key=None, suffix="")

In [ ]:
adata[adata.obs["cell_type_l1"] == "T"].obs["cell_type_l2"].value_counts()

In [ ]:
tcell_density = np.asarray(ad_tmp[:, "T"].X).flatten()

In [ ]:
# split into groups using jenkins natural breaks
n_groups = 5
jnb = JenksNaturalBreaks(n_groups)
jnb.fit(tcell_density)
groups = jnb.labels_.astype(str)

In [ ]:
adata.obs["tcell_density_group"] = groups
adata.obs["tcell_density"] = tcell_density

In [ ]:
for sample in adata.obs["sample"].unique()[1:2]:
    ad_sub = adata[adata.obs["sample"] == sample].copy()
    sc.pl.spatial(ad_sub, color=["tcell_density_group", "tcell_density"], spot_size=10)

### Using threshold

In [ ]:
threshold = 0.15
mask = tcell_density > threshold

adata.obs["tcell_infiltrate"] = "no infiltrate"
adata.obs.loc[mask, "tcell_infiltrate"] = "infiltrate"

In [ ]:
for sample in adata.obs["sample"].unique()[1:4]:
    ad_sub = adata[adata.obs["sample"] == sample].copy()
    sc.pl.spatial(
        ad_sub,
        color=["tcell_infiltrate", "tcell_density", "is_T"],
        spot_size=10,
        wspace=0,
    )

In [ ]:
adata.write_h5ad("data/xenium/processed/05.2-kidney_tcr_infilrate.h5ad")